# Experiment 006: Max Open Trades Sweep — Comparison

**Hypothesis:** `max_open_trades=10` artificially constrains the OI cap's dynamic sizing.
Since OI cap already limits position size per-pair, a higher trade count lets the strategy
diversify without overexposure. Exp 005 arm D had 361 rejected entries — some likely from
hitting the max trades ceiling.

**Base config:** Exp 005 arm D (all 107 pairs, OI cap only, no liq filter, no pool cap)

**Strategy params:** pool=1.0, oi=0.025, ratio=0.10, minpos=0.01

| Arm | max_open_trades | Description |
|-----|----------------|-------------|
| A (baseline) | 10 | Same as exp005 arm D |
| B | 15 | +50% trade slots |
| C | 20 | 2× trade slots |
| D | 30 | 3× trade slots |

In [1]:
import os, json, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/home/ubuntu/dev/gmx-ccxt-freqtrade')
os.chdir(PROJECT_ROOT)

STARTING_BALANCE = 100_000
TEMPLATE = 'plotly_dark'

RESULTS_DIR = PROJECT_ROOT / 'experiments/ichiv3-gmx/006-max-open-trades-sweep/results'

# Consistent colors across all charts
COLORS = {
    'Arm A (10 trades)': '#636EFA',
    'Arm B (15 trades)': '#00CC96',
    'Arm C (20 trades)': '#FFA15A',
    'Arm D (30 trades)': '#EF553B',
}

STYLES = {
    'Arm A (10 trades)': dict(color='#636EFA', width=3),
    'Arm B (15 trades)': dict(color='#00CC96', width=2),
    'Arm C (20 trades)': dict(color='#FFA15A', width=2),
    'Arm D (30 trades)': dict(color='#EF553B', width=2, dash='dash'),
}

In [2]:
def load_trades_from_zip(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        json_name = [n for n in z.namelist() if n.endswith('.json') and 'config' not in n and 'meta' not in n][0]
        with z.open(json_name) as f:
            data = json.load(f)
    strategy_name = list(data['strategy'].keys())[0]
    trades_list = data['strategy'][strategy_name]['trades']
    trades = pd.DataFrame(trades_list)
    trades['open_date'] = pd.to_datetime(trades['open_date'])
    trades['close_date'] = pd.to_datetime(trades['close_date'])
    return trades, strategy_name


def calculate_metrics(trades, starting_balance=STARTING_BALANCE):
    ts = trades.sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = starting_balance + ts['cum_profit_abs']

    total_profit_abs = ts['profit_abs'].sum()
    total_profit_pct = (total_profit_abs / starting_balance) * 100

    days = (ts['close_date'].max() - ts['open_date'].min()).days
    years = days / 365.25
    final_equity = starting_balance + total_profit_abs
    cagr = ((final_equity / starting_balance) ** (1 / years) - 1) * 100 if years > 0 else 0

    equity = ts['equity']
    rolling_max = equity.cummax()
    drawdown = (equity - rolling_max) / rolling_max * 100
    max_dd = drawdown.min()

    ts['close_day'] = ts['close_date'].dt.date
    daily_pnl = ts.groupby('close_day')['profit_abs'].sum()
    daily_returns = daily_pnl / starting_balance

    sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(365) if daily_returns.std() > 0 else 0
    downside = daily_returns[daily_returns < 0]
    sortino = (daily_returns.mean() / downside.std()) * np.sqrt(365) if len(downside) > 0 and downside.std() > 0 else 0
    calmar = abs(cagr / max_dd) if max_dd != 0 else 0

    win_rate = (ts['profit_abs'] > 0).mean() * 100

    return {
        'Trades': len(ts),
        'Rejected': None,  # filled in after loading
        'Profit ($)': round(total_profit_abs, 0),
        'Profit (%)': round(total_profit_pct, 1),
        'CAGR (%)': round(cagr, 1),
        'Max DD (%)': round(max_dd, 1),
        'Sharpe': round(sharpe, 2),
        'Sortino': round(sortino, 2),
        'Calmar': round(calmar, 2),
        'Win Rate (%)': round(win_rate, 1),
        'Avg Profit/Trade (%)': round(ts['profit_ratio'].mean() * 100, 2),
        'Avg Stake ($)': round(ts['stake_amount'].mean(), 0),
        'Med Stake ($)': round(ts['stake_amount'].median(), 0),
    }

In [3]:
# Load all 4 arms
configs = {
    'Arm A (10 trades)': RESULTS_DIR / 'arm_a_10trades.zip',
    'Arm B (15 trades)': RESULTS_DIR / 'arm_b_15trades.zip',
    'Arm C (20 trades)': RESULTS_DIR / 'arm_c_20trades.zip',
    'Arm D (30 trades)': RESULTS_DIR / 'arm_d_30trades.zip',
}

# Rejected entries from the backtest logs
rejected_entries = {
    'Arm A (10 trades)': 361,
    'Arm B (15 trades)': 62,
    'Arm C (20 trades)': 9,
    'Arm D (30 trades)': 0,
}

all_trades = {}
all_metrics = {}

for label, path in configs.items():
    trades, _ = load_trades_from_zip(path)
    all_trades[label] = trades
    metrics = calculate_metrics(trades)
    metrics['Rejected'] = rejected_entries[label]
    all_metrics[label] = metrics

df = pd.DataFrame(all_metrics).T
display_cols = ['Trades', 'Rejected', 'Profit ($)', 'CAGR (%)', 'Max DD (%)',
                'Sharpe', 'Sortino', 'Calmar', 'Win Rate (%)',
                'Avg Profit/Trade (%)', 'Avg Stake ($)', 'Med Stake ($)']
df[display_cols]

,Trades,Rejected,Profit ($),CAGR (%),Max DD (%),Sharpe,Sortino,Calmar,Win Rate (%),Avg Profit/Trade (%),Avg Stake ($),Med Stake ($)
Arm A (10 trades),1642.0,361.0,146460.0,21.7,-30.3,1.39,3.23,0.72,38.6,0.66,17291.0,17633.0
Arm B (15 trades),1884.0,62.0,139265.0,20.9,-24.3,1.54,4.03,0.86,38.9,0.65,11583.0,12163.0
Arm C (20 trades),2033.0,9.0,117545.0,18.4,-20.5,1.60,4.18,0.90,38.9,0.64,8670.0,9528.0
Arm D (30 trades),2121.0,0.0,91641.0,15.2,-16.9,1.69,4.60,0.90,39.1,0.66,6069.0,6595.0


## Equity Curves

In [4]:
fig = go.Figure()

for label, trades in all_trades.items():
    ts = trades.sort_values('close_date').copy()
    ts['equity'] = STARTING_BALANCE + ts['profit_abs'].cumsum()
    fig.add_trace(go.Scatter(x=ts['close_date'], y=ts['equity'], mode='lines',
                             name=label, line=STYLES[label]))

fig.update_layout(
    title='Equity Curves — Max Open Trades Sweep',
    xaxis_title='Date', yaxis_title='Equity ($)',
    template=TEMPLATE, height=600,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

## Drawdown Curves

In [5]:
fig = go.Figure()

for label, trades in all_trades.items():
    ts = trades.sort_values('close_date').copy()
    ts['equity'] = STARTING_BALANCE + ts['profit_abs'].cumsum()
    ts['drawdown'] = (ts['equity'] - ts['equity'].cummax()) / ts['equity'].cummax() * 100
    style = {k: v for k, v in STYLES[label].items() if k != 'width'}
    style['width'] = 1.5
    fig.add_trace(go.Scatter(x=ts['close_date'], y=ts['drawdown'], mode='lines',
                             name=label, line=style))

fig.update_layout(
    title='Drawdown Curves — Max Open Trades Sweep',
    xaxis_title='Date', yaxis_title='Drawdown (%)',
    template=TEMPLATE, height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

## Stake Size Distribution

In [6]:
buckets = [0, 500, 1000, 5000, 10000, 50000, float('inf')]
bucket_labels = ['<$500', '$500-1K', '$1K-5K', '$5K-10K', '$10K-50K', '>$50K']

dist_data = []
for label, trades in all_trades.items():
    for i in range(len(buckets)-1):
        count = len(trades[(trades['stake_amount'] >= buckets[i]) & (trades['stake_amount'] < buckets[i+1])])
        dist_data.append({'Config': label, 'Bucket': bucket_labels[i], 'Count': count})

dist_df = pd.DataFrame(dist_data)
fig = px.bar(dist_df, x='Bucket', y='Count', color='Config', barmode='group',
             title='Stake Size Distribution',
             template=TEMPLATE, height=500,
             color_discrete_map=COLORS,
             category_orders={'Bucket': bucket_labels})
fig.show()

## Parallelism & Capital Efficiency

In [7]:
def compute_daily_exposure(trades, starting_balance=STARTING_BALANCE):
    """For each day, compute open trade count and total capital deployed as % of equity."""
    ts = trades.copy()
    ts['open_date'] = pd.to_datetime(ts['open_date'])
    ts['close_date'] = pd.to_datetime(ts['close_date'])

    ts_sorted = ts.sort_values('close_date')
    ts_sorted['equity'] = starting_balance + ts_sorted['profit_abs'].cumsum()

    date_range = pd.date_range(ts['open_date'].min().normalize(),
                               ts['close_date'].max().normalize(), freq='D')

    equity_by_close = ts_sorted.groupby(ts_sorted['close_date'].dt.normalize())['equity'].last()
    equity_series = equity_by_close.reindex(date_range, method='ffill').fillna(starting_balance)

    records = []
    for day in date_range:
        open_mask = (ts['open_date'].dt.normalize() <= day) & (ts['close_date'].dt.normalize() >= day)
        open_trades = ts[open_mask]
        n_open = len(open_trades)
        total_deployed = open_trades['stake_amount'].sum()
        equity = equity_series.loc[day]
        pct_deployed = (total_deployed / equity * 100) if equity > 0 else 0
        records.append({'date': day, 'open_trades': n_open, 'deployed_pct': pct_deployed})

    return pd.DataFrame(records)


def hex_to_rgba(hex_color, alpha=0.3):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'


config_labels = list(all_trades.keys())
daily_exposure = {}
for label, trades in all_trades.items():
    daily_exposure[label] = compute_daily_exposure(trades)
    de = daily_exposure[label]
    print(f'{label}: avg open trades={de["open_trades"].mean():.1f}, '
          f'avg capital deployed={de["deployed_pct"].mean():.0f}%, '
          f'max open trades={de["open_trades"].max()}')

Arm A (10 trades): avg open trades=2.9, avg capital deployed=35%, max open trades=19


Arm B (15 trades): avg open trades=3.3, avg capital deployed=28%, max open trades=25


Arm C (20 trades): avg open trades=3.6, avg capital deployed=24%, max open trades=25


Arm D (30 trades): avg open trades=3.8, avg capital deployed=18%, max open trades=28


In [8]:
# Concurrent open trades over time
max_trades_map = {'Arm A (10 trades)': 10, 'Arm B (15 trades)': 15,
                  'Arm C (20 trades)': 20, 'Arm D (30 trades)': 30}

fig = make_subplots(rows=len(config_labels), cols=1, shared_xaxes=True,
                    subplot_titles=config_labels, vertical_spacing=0.06)

for i, label in enumerate(config_labels, 1):
    de = daily_exposure[label]
    fig.add_trace(go.Scatter(
        x=de['date'], y=de['open_trades'], mode='lines',
        name=label, line=dict(color=COLORS[label], width=1),
        fill='tozeroy', fillcolor=hex_to_rgba(COLORS[label], 0.3),
        showlegend=False,
    ), row=i, col=1)
    # Show the max_open_trades limit
    limit = max_trades_map[label]
    fig.add_hline(y=limit, line_dash='dot', line_color='red', line_width=0.8,
                  annotation_text=f'limit={limit}', annotation_font_color='red',
                  row=i, col=1)
    mean_val = de['open_trades'].mean()
    fig.add_hline(y=mean_val, line_dash='dash', line_color='white', line_width=0.5,
                  annotation_text=f'avg={mean_val:.1f}', annotation_font_color='white',
                  row=i, col=1)
    fig.update_yaxes(title_text='Open Trades', range=[0, 35], row=i, col=1)

fig.update_layout(template=TEMPLATE, height=220 * len(config_labels),
                  title='Parallelism — Concurrent Open Trades')
fig.show()

In [9]:
# Capital efficiency — % of equity deployed
fig2 = make_subplots(rows=len(config_labels), cols=1, shared_xaxes=True,
                     subplot_titles=config_labels, vertical_spacing=0.06)

for i, label in enumerate(config_labels, 1):
    de = daily_exposure[label]
    deployed_smooth = de['deployed_pct'].rolling(7, min_periods=1).mean()
    fig2.add_trace(go.Scatter(
        x=de['date'], y=deployed_smooth, mode='lines',
        name=label, line=dict(color=COLORS[label], width=1),
        fill='tozeroy', fillcolor=hex_to_rgba(COLORS[label], 0.3),
        showlegend=False,
    ), row=i, col=1)
    mean_val = de['deployed_pct'].mean()
    fig2.add_hline(y=mean_val, line_dash='dash', line_color='white', line_width=0.5,
                   annotation_text=f'avg={mean_val:.0f}%', annotation_font_color='white',
                   row=i, col=1)
    fig2.update_yaxes(title_text='Deployed %', range=[0, 120], row=i, col=1)

fig2.update_layout(template=TEMPLATE, height=220 * len(config_labels),
                   title='Capital Efficiency — % of Equity Deployed (7d smoothed)')
fig2.show()

## Profit per Trade — Marginal Value of Extra Slots

In [10]:
# Are the extra trades (unlocked by higher max_open_trades) profitable?
# Compare the marginal trades that only exist in higher-slot arms

# Profit per trade by arm
profit_per_trade = []
for label, trades in all_trades.items():
    profit_per_trade.append({
        'Arm': label,
        'Trades': len(trades),
        'Total Profit ($)': round(trades['profit_abs'].sum(), 0),
        'Avg Profit/Trade ($)': round(trades['profit_abs'].mean(), 2),
        'Median Profit/Trade ($)': round(trades['profit_abs'].median(), 2),
    })

ppt_df = pd.DataFrame(profit_per_trade).set_index('Arm')
display(ppt_df)

# Marginal analysis
arms = list(all_metrics.keys())
print('\nMarginal analysis (incremental trades vs incremental profit):')
for i in range(1, len(arms)):
    prev, curr = arms[i-1], arms[i]
    delta_trades = all_metrics[curr]['Trades'] - all_metrics[prev]['Trades']
    delta_profit = all_metrics[curr]['Profit ($)'] - all_metrics[prev]['Profit ($)']
    avg_marginal = delta_profit / delta_trades if delta_trades > 0 else 0
    print(f'  {prev} → {curr}: +{delta_trades} trades, '
          f'{delta_profit:+,.0f}$ profit, '
          f'avg marginal profit/trade: ${avg_marginal:,.0f}')

,Trades,Total Profit ($),Avg Profit/Trade ($),Median Profit/Trade ($)
Arm,,,,
Arm A (10 trades),1642,146460.0,89.20,-174.21
Arm B (15 trades),1884,139265.0,73.92,-116.54
Arm C (20 trades),2033,117545.0,57.82,-84.70
Arm D (30 trades),2121,91641.0,43.21,-65.38



Marginal analysis (incremental trades vs incremental profit):
  Arm A (10 trades) → Arm B (15 trades): +242 trades, -7,195$ profit, avg marginal profit/trade: $-30
  Arm B (15 trades) → Arm C (20 trades): +149 trades, -21,720$ profit, avg marginal profit/trade: $-146
  Arm C (20 trades) → Arm D (30 trades): +88 trades, -25,904$ profit, avg marginal profit/trade: $-294


## Monthly Returns Heatmap

In [11]:
fig = make_subplots(rows=2, cols=2, subplot_titles=list(all_trades.keys()),
                    vertical_spacing=0.12, horizontal_spacing=0.08)

positions = [(1,1), (1,2), (2,1), (2,2)]

for (row, col), (label, trades) in zip(positions, all_trades.items()):
    ts = trades.sort_values('close_date').copy()
    ts['month'] = ts['close_date'].dt.to_period('M')
    monthly = ts.groupby('month')['profit_abs'].sum()
    monthly_pct = (monthly / STARTING_BALANCE * 100).reset_index()
    monthly_pct.columns = ['month', 'return_pct']
    monthly_pct['year'] = monthly_pct['month'].dt.year
    monthly_pct['mon'] = monthly_pct['month'].dt.month

    pivot = monthly_pct.pivot(index='year', columns='mon', values='return_pct').fillna(0)
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]

    fig.add_trace(go.Heatmap(
        z=pivot.values, x=pivot.columns.tolist(), y=pivot.index.tolist(),
        colorscale='RdYlGn', zmid=0, showscale=(col == 2),
        text=np.round(pivot.values, 1), texttemplate='%{text}%',
    ), row=row, col=col)

fig.update_layout(template=TEMPLATE, height=700,
                  title='Monthly Returns (% of Starting Balance)')
fig.show()

## Summary

**Result:** The hypothesis was **rejected** — increasing `max_open_trades` *reduces* total profit.

| | 10 trades | 15 trades | 20 trades | 30 trades |
|---|---|---|---|---|
| **Total Profit** | $146K (best) | $139K | $118K | $92K |
| **Rejected Entries** | 361 | 62 | 9 | 0 |
| **Max % Underwater** | 30.3% | 24.3% | 20.5% | 16.9% |

**Key findings:**
- **Capital concentration wins.** Fewer trade slots → larger per-trade stakes → more profit from winners.
- **Marginal trades are negative EV.** The extra trades unlocked by higher limits dilute capital into worse opportunities.
- **Rejected entries are a feature, not a bug.** The 361 rejections at `max_open_trades=10` are the strategy correctly prioritizing its best positions.
- **Drawdown smoothing is real but costly.** Arm D (30) has the smoothest equity curve but at ~37% less total profit.

**Conclusion:** Keep `max_open_trades=10` as the production setting.